# San Ignacio de Velasco visto por NISAR — recorrido por los datos y primer análisis

**UraqiVision · NASA Space Apps 2026 · *Dancing with the SARs***

Este cuaderno abre todo lo que los scripts de descarga (`scripts/00`–`07`) dejaron en `data/` para la región que
escogimos, explica cada pieza y hace un primer análisis con las cuatro partes del MVP:

| Parte | Contenido |
|---|---|
| **A. Los datos** | 1. Dónde está la zona · 2. ¿Apuntan los datos al lugar correcto? · 3. Qué se descargó · 4. Cómo leer una imagen de radar · 5. La serie completa y qué fechas sirven · 6. La capa `mask` · 7. Focos de calor |
| **B. Análisis del MVP** | 8. Preparación y línea base · 9. Incendios y cicatrices de quema · 10. Desmonte o pérdida de estructura: clasificación honesta · 11. Recuperación y cambio de vegetación como contexto · 12. Validación limitada |
| **C. Cierre** | 13. Guía de imágenes raras · 14. Qué aprendimos |

> **Léelo antes de usar cualquier cifra.** Esto es una **exploración**, no el pipeline científico del proyecto
> (M-CIENCIA). Los umbrales **no están calibrados**, no hay polígonos de control (DP-05 pendiente) y la serie
> útil es muy corta. Los mapas y hectáreas de la parte B sirven para entender el método y sus límites; **no son
> resultados publicables** (RN-04, RN-05). Cada vez que un número es una regla práctica, el texto lo dice.

In [ ]:
from pathlib import Path
import json, warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio.windows import from_bounds
from rasterio import features
from pyproj import Transformer
from scipy import ndimage as ndi
from shapely.geometry import box, shape as geom_shape
import yaml
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Patch

warnings.filterwarnings("ignore", category=RuntimeWarning)   # log de cero y cortes vacíos

# Raíz del proyecto = carpeta que contiene config/aoi.yaml (funciona desde notebooks/ o desde la raíz)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "aoi.yaml").exists())
RAW, META = ROOT / "data" / "raw", ROOT / "data" / "meta"
cfg = yaml.safe_load(open(ROOT / "config" / "aoi.yaml", encoding="utf-8"))
manifest = json.load(open(META / "manifest.json", encoding="utf-8"))

W, S, E, N = (cfg["caja"][k] for k in ("oeste", "sur", "este", "norte"))
HUECO = (pd.Timestamp(cfg["periodo"]["hueco"]["inicio"]), pd.Timestamp(cfg["periodo"]["hueco"]["fin"]))
# Centro urbano de San Ignacio de Velasco: coordenadas APROXIMADAS (16°22′S, 60°58′O), solo para ubicarse
PUEBLO = (-60.960, -16.373)
MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
dm = lambda t: f"{pd.Timestamp(t).day} {MESES[pd.Timestamp(t).month - 1]}"       # '17 ago'

# Un solo lenguaje visual para todo el cuaderno
TINTA, TENUE, REJILLA = "#1f2933", "#6b7280", "#e5e7eb"
AZUL, NARANJA, VERDE, ROJO = "#2f6fb0", "#c8641e", "#3d8a5f", "#b3392f"
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": REJILLA, "axes.labelcolor": TENUE, "axes.titlecolor": TINTA,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": TENUE, "ytick.color": TENUE, "text.color": TINTA,
    "grid.color": REJILLA, "legend.frameon": False, "font.size": 10,
})
# Mapa divergente para el cambio: pérdida (cálido) – gris neutro – aumento (frío)
CAMBIO = LinearSegmentedColormap.from_list("cambio", ["#8c2d04", "#d9793a", "#ececec", "#6aa3cf", "#1d4f85"])
CAMBIO.set_bad("#bdbdbd")
GRIS = plt.get_cmap("gray").copy(); GRIS.set_bad("#f3d9c9")   # sin dato = naranja pálido, nunca negro
sin_ejes = lambda ax: (ax.set_xticks([]), ax.set_yticks([]))

print("Raíz del proyecto:", ROOT)
print(f"Caja de estudio: longitud {W} a {E}, latitud {S} a {N}")

# A. Los datos

## 1. ¿Dónde está la zona de estudio?

La caja se define una sola vez, en `config/aoi.yaml`; todos los scripts la leen de ahí. Rodea al pueblo de
San Ignacio de Velasco, en la Chiquitanía (departamento de Santa Cruz).

NISAR vuela trayectorias fijas llamadas **órbitas (tracks)**, y cada una se corta en **frames**. Nuestra serie
usa la **órbita 060 ascendente**. La caja cae sobre dos frames: el **169** la cubre casi entera y el **170** solo
la franja norte.

In [ ]:
deps = gpd.read_file(RAW / "boundaries" / "bolivia_adm1.geojson")
huellas = gpd.read_file(META / "nisar_huellas_bolivia.geojson")
t060 = huellas[(huellas["track"] == 60) & (huellas["direccion"] == "A") & (huellas["frame"].isin([169, 170]))].drop_duplicates("frame")
caja = gpd.GeoDataFrame(geometry=[box(W, S, E, N)], crs=4326)

fig, (a, b) = plt.subplots(1, 2, figsize=(12, 6))
deps.plot(ax=a, color="#f4f5f7", edgecolor="#9aa3ad", linewidth=0.7)
deps[deps["shapeName"] == "Santa Cruz"].plot(ax=a, color="#dde6f0", edgecolor="#9aa3ad", linewidth=0.7)
t060.plot(ax=a, facecolor=AZUL, alpha=0.18, edgecolor=AZUL, linewidth=1.2)
caja.plot(ax=a, facecolor="none", edgecolor=NARANJA, linewidth=2)
for _, r in deps.iterrows():
    p = r.geometry.representative_point()
    a.annotate(r["shapeName"], (p.x, p.y), ha="center", fontsize=8, color=TENUE)
a.set_title("Bolivia: frames de NISAR y caja de estudio")
a.legend(handles=[Patch(facecolor=AZUL, alpha=0.3, edgecolor=AZUL, label="NISAR órbita 060A, frames 169 y 170"),
                  Patch(facecolor="none", edgecolor=NARANJA, linewidth=2, label="Caja de estudio")], loc="lower left")

deps.plot(ax=b, color="#f4f5f7", edgecolor="#9aa3ad", linewidth=0.7)
for _, r in t060.iterrows():
    gpd.GeoSeries([r.geometry], crs=4326).plot(ax=b, facecolor=AZUL, alpha=0.12, edgecolor=AZUL, linewidth=1.2)
    c = r.geometry.intersection(box(W - 0.9, S - 0.9, E + 0.9, N + 0.9)).representative_point()
    b.annotate(f"frame {r['frame']}", (c.x, c.y), color=AZUL, fontsize=9, ha="center")
caja.plot(ax=b, facecolor="none", edgecolor=NARANJA, linewidth=2)
b.plot(*PUEBLO, marker="*", color=TINTA, markersize=13, linestyle="none")
b.annotate("San Ignacio\nde Velasco", PUEBLO, xytext=(8, -4), textcoords="offset points", fontsize=9)
b.annotate("caja de estudio", (W, N + 0.03), color=NARANJA, fontsize=9)
b.set_xlim(W - 1, E + 1); b.set_ylim(S - 1, N + 1)
b.set_title("Acercamiento: el pueblo queda cerca del centro de la caja")
for ax in (a, b):
    ax.set_xlabel("longitud"); ax.set_ylabel("latitud"); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

utm = Transformer.from_crs(4326, 32721, always_xy=True)
(x0, y0), (x1, y1) = utm.transform(W, S), utm.transform(E, N)
print(f"Tamaño de la caja: {abs(x1 - x0) / 1000:.1f} km (este-oeste) × {abs(y1 - y0) / 1000:.1f} km (norte-sur)")
print(f"El pueblo está a {(PUEBLO[0] - W) / (E - W) * 100:.0f}% del borde oeste y a {(PUEBLO[1] - S) / (N - S) * 100:.0f}% del borde sur de la caja.")

## 2. ¿Apuntan los datos al lugar correcto?

Una imagen de radar no trae nombres ni calles, así que a simple vista cuesta saber si «eso» es San Ignacio.
Hay dos formas de comprobarlo con lo que tenemos.

**Comprobación 1 · el pueblo.** Recortamos 14 × 14 km alrededor de las coordenadas del centro urbano y miramos
tres imágenes independientes:

- **HH del radar**: las ciudades brillan mucho, porque las paredes y el suelo forman esquinas que devuelven la señal
  (doble rebote).
- **HV del radar**: resalta la vegetación; una ciudad se ve intermedia y el agua, negra.
- **Sentinel-2, infrarrojo de onda corta (SWIR)**: techos, calles y suelo desnudo se ven claros; el agua, negra.

Si los datos están bien ubicados, las tres deben mostrar una mancha urbana en el mismo sitio, junto a la
estrella.

In [ ]:
FECHAS = sorted({v["fecha"] for v in manifest["nisar"].values()})           # 'AAAA-MM-DD'
NISAR_DIR = RAW / "nisar_provisional"
REF = "2026-08-17"                                                          # fecha limpia de referencia

def ruta(fecha, capa, frame=169):
    return NISAR_DIR / f"{cfg['nombre']}_{fecha.replace('-', '')}_T060A_F{frame}_{capa}.tif"

def leer(fecha, capa, frame=169, factor=1):
    # Lee una capa; con factor>1 promedia factor×factor píxeles en potencia lineal (multilook)
    with rasterio.open(ruta(fecha, capa, frame)) as ds:
        h, w = ds.height // factor, ds.width // factor
        a = ds.read(1, out_shape=(h, w), resampling=Resampling.average if factor > 1 else Resampling.nearest)
        tr = ds.transform * ds.transform.scale(ds.width / w, ds.height / h)
        return a.astype("float32"), tr, ds.crs

def db(lineal):
    return np.where(lineal > 0, 10 * np.log10(np.where(lineal > 0, lineal, 1)), np.nan)

def extension(tr, shape):
    return [tr.c, tr.c + tr.a * shape[1], tr.f + tr.e * shape[0], tr.f]

def recorte(archivo, crs_destino, radio=7000):
    # Ventana cuadrada de 2·radio metros centrada en el pueblo, a resolución completa
    with rasterio.open(archivo) as ds:
        px, py = Transformer.from_crs(4326, ds.crs, always_xy=True).transform(*PUEBLO)
        win = from_bounds(px - radio, py - radio, px + radio, py + radio, ds.transform)
        return ds.read(1, window=win).astype("float32"), [px - radio, px + radio, py - radio, py + radio], (px, py)

s2 = manifest["sentinel2"]
hh_p, ext_p, c_p = recorte(ruta(REF, "HHHH"), 32721)
hv_p, _, _ = recorte(ruta(REF, "HVHV"), 32721)
sw_p, ext_s, c_s = recorte(ROOT / s2["antes"]["rutas"]["swir22"], 32720)

fig, axs = plt.subplots(1, 3, figsize=(15, 5.6))
for ax, img, e, c, tit in zip(axs, [db(hh_p), db(hv_p), sw_p / 1e4], [ext_p, ext_p, ext_s], [c_p, c_p, c_s],
        [f"Radar HH · {dm(REF)} (ciudad = brillante)", f"Radar HV · {dm(REF)} (vegetación = brillante)",
         f"Sentinel-2 SWIR · {dm(s2['antes']['fecha'])} (techos y suelo = claro)"]):
    lo, hi = np.nanpercentile(img, [2, 98])
    ax.imshow(img, cmap=GRIS, vmin=lo, vmax=hi, extent=e, interpolation="nearest")
    ax.plot(*c, marker="*", color="#ffd400", markeredgecolor=TINTA, markersize=17, linestyle="none")
    ax.set_title(tit); sin_ejes(ax)
fig.suptitle("14 × 14 km alrededor de las coordenadas de San Ignacio de Velasco (estrella)", x=0.01, ha="left", fontsize=12)
plt.tight_layout(); plt.show()

**Comprobación 2 · radar contra óptico en toda la caja.** NISAR y Sentinel-2 son satélites distintos y cada uno
se georreferencia por su cuenta. Si llevamos el índice de vegetación óptico (NBR) a la grilla del radar, el
bosque (HV alto, NBR alto) y los campos (HV bajo, NBR bajo) deben coincidir **sin mover nada**. La celda busca
el desplazamiento que mejor los alinea: si los datos están en su sitio, sale cero.

In [ ]:
hh, tr3, crs169 = leer(REF, "HHHH", factor=3)                 # 20 m → 60 m
hv, _, _ = leer(REF, "HVHV", factor=3)
hh_db, hv_db = db(hh), db(hv)
ext = extension(tr3, hh.shape)

def leer_s2(cual):
    r = s2[cual]["rutas"]; out = {}
    for b in ("nir08", "swir22", "scl"):
        with rasterio.open(ROOT / r[b]) as ds:
            out[b] = ds.read(1); out["tr"], out["crs"] = ds.transform, ds.crs
    nir, swir = out["nir08"] / 1e4, out["swir22"] / 1e4            # reflectancia = DN / 10000 (offset ya aplicado)
    valido = np.isin(out["scl"], [4, 5, 6, 7, 11]) & (out["nir08"] > 0)   # vegetación, suelo, agua, sin clasificar
    out["nbr"] = np.where(valido & (nir + swir > 0), (nir - swir) / (nir + swir), np.nan)
    return out

def a_grilla_radar(a, src):
    out = np.full(hh.shape, np.nan, dtype="float32")
    reproject(a.astype("float32"), out, src_transform=src["tr"], src_crs=src["crs"], src_nodata=np.nan,
              dst_transform=tr3, dst_crs=crs169, dst_nodata=np.nan, resampling=Resampling.average)
    return out

antes, despues = leer_s2("antes"), leer_s2("despues")
nbr_antes60, nbr_desp60 = a_grilla_radar(antes["nbr"], antes), a_grilla_radar(despues["nbr"], despues)

def desplazamiento(a, b):
    # Desplazamiento (filas, columnas) que mejor alinea b con a, por correlación de fase
    prep = lambda x: np.nan_to_num(x - np.nanmean(x))
    F = np.fft.fft2(prep(a)) * np.conj(np.fft.fft2(prep(b)))
    c = np.fft.ifft2(F / (np.abs(F) + 1e-9)).real
    i, j = np.unravel_index(c.argmax(), c.shape)
    return (i if i < c.shape[0] // 2 else i - c.shape[0]), (j if j < c.shape[1] // 2 else j - c.shape[1])

def correlacion(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.corrcoef(a[m], b[m])[0, 1]) if m.sum() > 1000 else np.nan

hv_rec = np.clip(hv_db, -25, -5)
di0, dj0 = desplazamiento(hv_rec, nbr_desp60)
print(f"Desplazamiento que mejor alinea el NBR óptico con el HV del radar: {di0 * 60} m (norte-sur), {dj0 * 60} m (este-oeste)")
print(f"Correlación HV ({dm(REF)}) contra NBR ({dm(s2['despues']['fecha'])}), sin mover nada: {correlacion(hv_rec, nbr_desp60):+.2f}")

**Lectura: sí, los datos apuntan a San Ignacio de Velasco.**

- En las tres imágenes del pueblo se reconoce lo mismo junto a la estrella: la **cuadrícula de calles** del
  casco urbano y, pegada a su lado noreste, una **mancha negra alargada que es agua** (la represa del pueblo).
  Radar y óptico la dibujan con la misma forma y en el mismo sitio.
- En toda la caja, el desplazamiento que mejor alinea óptico y radar es **0 m**, con una correlación de **+0,57**
  entre dos magnitudes distintas (HV y NBR) tomadas con seis semanas de diferencia.

Es decir, el radar de las fechas limpias y la imagen óptica muestran el mismo paisaje en el mismo sitio.

Dos avisos honestos:

- Las coordenadas del pueblo son aproximadas; sirven para ubicarse, no para medir.
- Esta comprobación vale para las fechas limpias. Más abajo (sección 5b) se ve que **una fecha, el 30 de junio,
  no está en su sitio**.

## 3. ¿Qué se descargó?

La tabla sale de `data/meta/manifest.json`, que los scripts escriben al avanzar. Los archivos de NISAR **nunca
se descargaron enteros** (pesan unos 2 GB cada uno): el script `02` abre el archivo remoto y lee solo los
píxeles de la caja.

In [ ]:
nis = pd.DataFrame(manifest["nisar"].values())
fi = manifest["firms"]["bolivia_2026"]
gb = lambda p: sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9
resumen = pd.DataFrame([
    ["NISAR L2 GCOV (PROVISIONAL)", f"órbita 060A · {nis['fecha'].nunique()} fechas · frames {sorted(int(x) for x in nis['frame'].unique())} · "
     "capas HHHH, HVHV, mask, numberOfLooks", f"{nis['fecha'].min()} → {nis['fecha'].max()}", f"{gb(RAW/'nisar_provisional'):.2f} GB"],
    ["Sentinel-2 L2A", "bandas nir08, swir22 y clasificación de escena (scl), 20 m",
     f"antes {s2['antes']['fecha']} · después {s2['despues']['fecha']}", f"{gb(RAW/'sentinel2'):.2f} GB"],
    ["Focos de calor FIRMS", f"{fi['focos']:,} en Bolivia, {fi['focos_caja']:,} dentro de la caja (VIIRS y MODIS)",
     f"{fi['desde']} → {fi['hasta']}", f"{gb(RAW/'firms'):.2f} GB"],
    ["Límites departamentales", "9 departamentos (geoBoundaries ADM1)", "edición 2015", f"{gb(RAW/'boundaries'):.2f} GB"],
], columns=["Dato", "Contenido", "Fechas", "Peso"])
display(resumen.style.hide(axis="index").set_properties(**{"text-align": "left"}))
print(f"Total en data/: {gb(ROOT/'data'):.2f} GB (límite en la configuración: {cfg['limite_gb']} GB)")

## 4. Cómo leer una imagen de radar

El radar de banda L de NISAR emite sus propios pulsos de microondas (unos 24 cm de longitud de onda) y mide
cuánto vuelve, así que funciona de noche y a través del humo y las nubes. El valor de cada píxel es la
**retrodispersión en potencia lineal** (gamma-cero, γ⁰). Aquí importan dos canales:

| Canal | Significado | Brilla donde… |
|---|---|---|
| **HH** | emite horizontal, recibe horizontal | hay superficies rugosas, y troncos o paredes sobre suelo o agua (doble rebote) |
| **HV** | emite horizontal, recibe vertical | hay **volumen de vegetación**: ramas y troncos dispersan la señal. **Baja cuando se pierde el bosque** |

Como los valores abarcan varios órdenes de magnitud, se *miran* en **decibelios**: `dB = 10·log10(potencia)`.
Los promedios siempre se calculan primero en potencia lineal y se convierten después.

**Por qué la imagen parece «arenosa».** El grano fino que se ve en todas las imágenes de radar se llama
**speckle**. No es suciedad ni un error de descarga: es propio de cómo se forma la imagen. Por eso se promedian
píxeles vecinos (aquí, 3 × 3: de 20 m a 60 m) antes de comparar fechas.

Abajo: una fecha limpia (17 ago 2026), frame 169. En HV, las manchas oscuras de bordes rectos son tierra
desmontada, cultivos y pasturas; las zonas claras y texturadas son bosque.

In [ ]:
ratio_db = hv_db - hh_db
fig, axs = plt.subplots(1, 3, figsize=(15, 5.4))
for ax, img, tit, lab in zip(axs, [hh_db, hv_db, ratio_db],
        ["HH — superficies y doble rebote", "HV — volumen de vegetación", "Cociente HV/HH — cuánta señal es volumen"],
        ["HH (dB)", "HV (dB)", "HV − HH (dB)"]):
    lo, hi = np.nanpercentile(img, [2, 98])
    im = ax.imshow(img, cmap=GRIS, vmin=lo, vmax=hi, extent=ext, interpolation="nearest")
    ax.set_title(tit); sin_ejes(ax)
    fig.colorbar(im, ax=ax, orientation="horizontal", pad=0.03, shrink=0.8, label=lab)
fig.suptitle(f"NISAR banda L, {REF}, órbita 060A frame 169, promediado a 60 m", x=0.01, ha="left", fontsize=12)
plt.tight_layout(); plt.show()
print(f"Grilla: {hh.shape[1]}×{hh.shape[0]} píxeles de 60 m, CRS {crs169}")

Una forma habitual de ver los dos canales a la vez es una **composición en falso color**: rojo = HH,
verde = HV, azul = HH/HV. El bosque sale verde (HV fuerte); la tierra abierta o desmontada, entre morado y
rosa (domina HH); el agua y las superficies muy lisas, oscuras.

Dos cosas que pueden parecer raras y no lo son:

- **La cuña clara de la esquina superior izquierda** es el borde del frame 169: ahí este frame no tiene datos
  (esa franja la cubre el frame 170). Se dibuja en color pálido para no confundirla con un valor.
- **Los campos a rayas** (franjas paralelas claras y oscuras) son parcelas agrícolas con franjas de bosque
  dejadas entre ellas. Es una *interpretación* del patrón, no algo verificado en terreno.

In [ ]:
def estirar(x, p=(2, 98)):
    lo, hi = np.nanpercentile(x, p)
    return np.clip((x - lo) / (hi - lo), 0, 1)

rgb = np.dstack([estirar(hh_db), estirar(hv_db), estirar(hh_db - hv_db)])
rgb = np.where(np.isnan(rgb), 1, rgb)
px60, py60 = Transformer.from_crs(4326, crs169, always_xy=True).transform(*PUEBLO)
fig, ax = plt.subplots(figsize=(8.5, 8.5))
ax.imshow(rgb, extent=ext, interpolation="nearest")
ax.plot(px60, py60, marker="*", color="#ffd400", markeredgecolor=TINTA, markersize=16, linestyle="none")
ax.annotate("San Ignacio de Velasco", (px60, py60), xytext=(10, 8), textcoords="offset points", color="white", fontweight="bold")
ax.set_title(f"Falso color (R = HH, G = HV, B = HH/HV) · {REF}")
ax.set_xlabel(f"este, m ({crs169})"); ax.set_ylabel("norte, m"); ax.ticklabel_format(style="plain")
plt.tight_layout(); plt.show()

## 5. La serie completa: ¿en qué fechas se puede confiar?

NISAR vuelve a pasar cada 12 días. Entre el **27 de julio y el 10 de agosto de 2026 el instrumento tiene un
hueco permanente de datos**, que parte la serie en dos. Aquí están las nueve fechas de HV, dibujadas con la
**misma escala de grises** para poder compararlas a ojo.

**Estas son las imágenes que «no representan nada».** Cuatro de las nueve se ven mal, y cada una por un motivo
distinto:

| Fecha | Cómo se ve | Por qué |
|---|---|---|
| 18 jun | casi toda blanca, con una banda oscura | Los valores son unas 30 veces más altos de lo normal y saturan la escala. La imagen de vista previa de la propia NASA muestra las mismas bandas: el defecto viene del producto PROVISIONAL, no de nuestra descarga |
| 30 jun | parece normal, pero borrosa | Los valores son correctos, pero el paisaje está **desplazado unos 20 km** (sección 5b) |
| 12 jul | un solo color plano | El archivo trae **ceros en todos los píxeles**: no hay imagen |
| 24 jul | casi toda blanca | Valores unas 8 veces más altos de lo normal en toda la escena |

Las cinco fechas posteriores al hueco se ven iguales entre sí, que es lo esperable: el paisaje casi no cambia en
12 días.

In [ ]:
ver = pd.read_csv(META / "verificacion.csv", dtype={"fecha": str, "frame": str})
ver["fecha"] = pd.to_datetime(ver["fecha"])
v169 = ver[(ver["frame"] == "169") & ver["capa"].isin(["HHHH", "HVHV"])]

pila = {f: leer(f, "HVHV", factor=6)[0] for f in FECHAS}      # 120 m: liviano para nueve paneles
parecido = lambda a, b: correlacion(np.clip(db(a), -25, -5), np.clip(db(b), -25, -5))
PARECIDO = {f: parecido(pila[REF], pila[f]) for f in FECHAS}

# Veredicto por fecha (frame 169): los valores los revisa scripts/07_verificar.py; la posición, la correlación
# con la fecha limpia de referencia (el paisaje casi no cambia en semanas, así que debería rondar 1).
def veredicto(f):
    notas = " ".join(v169[v169["fecha"] == pd.Timestamp(f)]["notas"].fillna("").astype(str))
    p50 = v169[(v169["fecha"] == pd.Timestamp(f)) & (v169["capa"] == "HVHV")]["p50"].iloc[0]
    if p50 == 0: return "vacía"
    if "mediana" in notas: return "valores malos"
    if PARECIDO[f] < 0.5: return "desplazada"
    return "buena"
VERED = {f: veredicto(f) for f in FECHAS}
COLOR_V = {"buena": VERDE, "desplazada": ROJO, "valores malos": ROJO, "vacía": ROJO}
BUENAS = [f for f in FECHAS if VERED[f] == "buena"]
lo, hi = np.nanpercentile(np.concatenate([db(pila[f]).ravel() for f in BUENAS]), [2, 98])

fig, axs = plt.subplots(2, 5, figsize=(16, 7))
for ax, f in zip(axs.ravel(), FECHAS):
    ax.imshow(db(pila[f]), cmap=GRIS, vmin=lo, vmax=hi, interpolation="nearest")
    ax.set_title(f, color=TINTA); sin_ejes(ax)
    ax.text(0.02, 0.03, VERED[f], transform=ax.transAxes, color="white", fontsize=9, fontweight="bold",
            bbox=dict(facecolor=COLOR_V[VERED[f]], edgecolor="none", pad=3))
    for s in ax.spines.values(): s.set_visible(False)
axs.ravel()[-1].axis("off")
axs.ravel()[-1].text(0, 0.5, f"Hueco de datos entre el\n{dm(HUECO[0])} y el {dm(HUECO[1])}:\ncuatro fechas antes,\ncinco después.\n\n"
                     "Naranja pálido = sin dato\n(ceros o fuera del frame)\n\nBlanco total = valores\nfuera de la escala", va="center", color=TENUE, fontsize=10)
fig.suptitle(f"HV en cada fecha, frame 169, una sola escala de grises ({lo:.0f} a {hi:.0f} dB)", x=0.01, ha="left", fontsize=12)
plt.tight_layout(); plt.show()

Lo mismo en números: la mediana de retrodispersión de toda la caja en cada fecha. Una serie sana debe ser casi
plana.

In [ ]:
serie = v169.pivot(index="fecha", columns="capa", values="p50").rename(columns={"HHHH": "HH", "HVHV": "HV"})
serie_db = 10 * np.log10(serie.where(serie > 0))
ok = pd.Series({pd.Timestamp(f): VERED[f] == "buena" for f in FECHAS})

fig, ax = plt.subplots(figsize=(11, 4.6))
ax.axvspan(*HUECO, color="#f1f1f1"); ax.text(HUECO[0] + (HUECO[1] - HUECO[0]) / 2, 9.2, "hueco de datos", ha="center", color=TENUE, fontsize=9)
for col, c in [("HH", AZUL), ("HV", NARANJA)]:
    y = serie_db[col]
    ax.plot(y[ok].index, y[ok], color=c, linewidth=2, marker="o", markersize=7, label=f"{col}, fechas buenas")
    ax.scatter(y[~ok].index, y[~ok], s=70, facecolors="white", edgecolors=c, linewidths=2, zorder=3, label=f"{col}, fechas excluidas")
    ax.annotate(col, (y[ok].index[-1], y[ok].iloc[-1]), xytext=(8, 0), textcoords="offset points", color=TINTA, va="center")
for f in [pd.Timestamp(f) for f in FECHAS if VERED[f] == "vacía"]:
    ax.annotate("12 jul: vacía\n(todo ceros)", (f, -14.5), ha="center", color=TENUE, fontsize=9)
ax.set_ylabel("mediana de retrodispersión de la caja (dB)"); ax.set_ylim(-15.5, 10.5); ax.grid(axis="y")
ax.set_xticks([pd.Timestamp(f) for f in FECHAS]); ax.set_xticklabels([dm(f) for f in FECHAS], rotation=45, ha="right")
ax.set_title("Mediana por fecha, frame 169: plana después del hueco, errática antes")
ax.legend(ncol=2, loc="upper right"); plt.tight_layout(); plt.show()

tabla = serie_db.round(1).assign(parecido=[round(PARECIDO[f.strftime('%Y-%m-%d')], 2) for f in serie_db.index],
                                 veredicto=[VERED[f.strftime('%Y-%m-%d')] for f in serie_db.index])
tabla.index = tabla.index.strftime("%Y-%m-%d"); tabla.index.name = "fecha"
tabla.columns = ["mediana HH (dB)", "mediana HV (dB)", f"correlación con {REF}", "veredicto"]
display(tabla)

### 5b. Por qué se rechaza el 30 de junio: está en el lugar equivocado

La escena del 30 de junio pasa todas las pruebas de valores, así que tienta usarla como imagen de «antes». Pero
su correlación con la referencia limpia es prácticamente cero: los campos y los bordes del bosque no coinciden.

La celda busca el desplazamiento que mejor la alinea con el 17 de agosto. Si solo estuviera borrosa, el mejor
desplazamiento sería cero.

In [ ]:
MALA = "2026-06-30"
mala60 = leer(MALA, "HVHV", factor=3)[0]
prep = lambda a: np.clip(db(a), -25, -5)
di, dj = desplazamiento(prep(hv), prep(mala60))
movida = np.roll(mala60, (di, dj), (0, 1))
real = np.ones_like(movida, bool)                              # los píxeles que «dieron la vuelta» no son dato real
if di < 0: real[di:, :] = False
elif di > 0: real[:di, :] = False
if dj < 0: real[:, dj:] = False
elif dj > 0: real[:, :dj] = False
movida = np.where(real, movida, np.nan)

lo2, hi2 = np.nanpercentile(hv_db, [2, 98])
fig, axs = plt.subplots(1, 3, figsize=(15, 5.6))
for ax, img, tit in zip(axs, [hv, mala60, movida],
        [f"{REF} (referencia limpia)", f"{MALA} tal como llegó · correlación {parecido(hv, mala60):+.2f}",
         f"{MALA} desplazada · correlación {parecido(hv, movida):+.2f}"]):
    ax.imshow(db(img), cmap=GRIS, vmin=lo2, vmax=hi2, interpolation="nearest"); ax.set_title(tit); sin_ejes(ax)
fig.suptitle("HV, frame 169: la escena del 30 de junio solo se parece al paisaje después de moverla", x=0.01, ha="left", fontsize=12)
plt.tight_layout(); plt.show()
print(f"Mejor desplazamiento: {abs(di) * 60 / 1000:.1f} km hacia el {'norte' if di < 0 else 'sur'} y "
      f"{abs(dj) * 60 / 1000:.1f} km hacia el {'oeste' if dj < 0 else 'este'} (grilla de 60 m).")
print(f"Las fechas limpias correlacionan entre {min(PARECIDO[f] for f in BUENAS):.2f} y 1.00 con la referencia sin mover nada.")

**Lectura.** Después del desplazamiento, la escena del 30 de junio coincide con el paisaje: su contenido es
real, pero está **georreferenciada a unos 20 km de donde debería**. Además se ve más borrosa que las fechas
limpias. Tal como llegó, esta fecha se excluye, con su motivo, igual que las otras tres.

La versión desplazada es un diagnóstico, no una corrección: un desplazamiento de píxeles enteros es el modelo
más simple posible y nada aquí prueba que el error sea una traslación pura. Si una versión realineada pudiera
servir de «antes» aproximado es una decisión del equipo (sería un producto modificado y habría que
documentarlo), no algo que adoptar en silencio.

**Consecuencia para la ciencia.** En esta órbita **no queda ninguna fecha utilizable antes del hueco**. La
línea base del método (MC-02) solo puede construirse con fechas posteriores al 10 de agosto.

## 6. La capa `mask`

Cada escena trae un ráster `mask` que marca píxeles. **Qué significa cada valor sigue siendo una decisión
pendiente del proyecto (DP-03)**, así que aquí solo se describe lo observado: `1` sobre datos normales, `255`
fuera del frame y `0` sobre las partes dañadas de las fechas malas.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4.9))
cm = ListedColormap([ROJO, "#dfe7ef", "#f3d9c9"])
for ax, f in zip(axs, ["2026-06-18", "2026-07-12", "2026-08-17"]):
    with rasterio.open(ruta(f, "mask")) as ds:
        m = ds.read(1, out_shape=(ds.height // 6, ds.width // 6), resampling=Resampling.nearest)
    ax.imshow(np.select([m == 0, m == 1], [0, 1], 2), cmap=cm, vmin=0, vmax=2, interpolation="nearest")
    partes = ", ".join(f"{v}: {100 * (m == v).mean():.0f}%" for v in np.unique(m))
    ax.set_title(f"{f}  ({VERED[f]})"); ax.set_xlabel(f"valores de mask → {partes}"); sin_ejes(ax)
axs[0].legend(handles=[Patch(color=ROJO, label="0"), Patch(color="#dfe7ef", label="1"), Patch(color="#f3d9c9", label="255")],
              title="valor de mask", loc="lower left", frameon=True, facecolor="white", edgecolor=REJILLA)
fig.suptitle("La máscara en dos fechas malas y una buena, frame 169", x=0.01, ha="left", fontsize=12)
plt.tight_layout(); plt.show()

La máscara detecta el daño del 18 de junio (43 % marcado) y del 12 de julio (100 %), así que sirve de filtro.
**No** marca el 24 de julio (valores malos en toda la escena) ni el 30 de junio (desplazada): esas fechas hay
que rechazarlas por sus valores y por su posición.

## 7. Focos de calor (FIRMS)

FIRMS publica los puntos donde los sensores VIIRS y MODIS detectaron calor. Dicen *cuándo* algo ardía, cosa
que el radar no puede. En el proyecto se usan para **fechar y clasificar** los parches de cambio, nunca para
crearlos (MC-04).

Al leer los conteos: pasan tres instrumentos cada día, así que un mismo incendio produce varios puntos.

In [ ]:
firms = pd.read_csv(RAW / "firms" / f"{cfg['nombre']}_2026.csv", parse_dates=["acq_date"])
diario = firms.groupby("acq_date").size().reindex(pd.date_range(firms["acq_date"].min(), firms["acq_date"].max()), fill_value=0)

fig, ax = plt.subplots(figsize=(12, 4.4))
ax.axvspan(*HUECO, color="#f1f1f1")
ax.bar(diario.index, diario.values, width=0.9, color=NARANJA)
for f in FECHAS:
    ax.axvline(pd.Timestamp(f), color=COLOR_V[VERED[f]], linewidth=1.3, linestyle="-" if VERED[f] == "buena" else ":")
pico = diario.idxmax()
ax.annotate(f"{dm(pico)}: {diario.max()} focos", (pico, diario.max()), xytext=(8, -4), textcoords="offset points", color=TINTA)
ax.text(HUECO[0] + (HUECO[1] - HUECO[0]) / 2, diario.max() * 0.9, "hueco\ndel radar", ha="center", color=TENUE, fontsize=9)
ax.set_ylabel("focos por día dentro de la caja"); ax.grid(axis="y")
ticks = pd.date_range("2026-07-01", diario.index.max(), freq="MS")
ax.set_xticks(ticks); ax.set_xticklabels([dm(t) for t in ticks])
ax.set_title("Focos por día en la caja, con las fechas del radar como líneas verticales")
ax.legend(handles=[plt.Line2D([], [], color=VERDE, label="fecha de radar buena"),
                   plt.Line2D([], [], color=ROJO, linestyle=":", label="fecha de radar excluida")], loc="upper left")
plt.tight_layout(); plt.show()

PRIMERA, ULTIMA = pd.Timestamp(BUENAS[0]), pd.Timestamp(BUENAS[-1])
firms["periodo"] = np.select([firms["acq_date"] < PRIMERA, firms["acq_date"] <= ULTIMA],
                             [f"antes del {dm(PRIMERA)}", f"{dm(PRIMERA)} – {dm(ULTIMA)}"], f"después del {dm(ULTIMA)}")
PERIODOS = [f"antes del {dm(PRIMERA)}", f"{dm(PRIMERA)} – {dm(ULTIMA)}", f"después del {dm(ULTIMA)}"]
display(firms.groupby("periodo").size().reindex(PERIODOS).rename("focos").to_frame())

In [ ]:
a_utm = Transformer.from_crs(4326, crs169, always_xy=True)
firms["x"], firms["y"] = a_utm.transform(firms["longitude"].values, firms["latitude"].values)
firms["col"] = ((firms["x"] - tr3.c) / tr3.a).astype(int); firms["fil"] = ((firms["y"] - tr3.f) / tr3.e).astype(int)

fig, ax = plt.subplots(figsize=(9, 8.6))
ax.imshow(hv_db, cmap=GRIS, vmin=lo2, vmax=hi2, extent=ext, alpha=0.75)
for per, c in zip(PERIODOS, [AZUL, NARANJA, VERDE]):
    d = firms[firms["periodo"] == per]
    ax.scatter(d["x"], d["y"], s=14, color=c, edgecolors="white", linewidths=0.4, label=f"{per} ({len(d)})")
ax.plot(px60, py60, marker="*", color="#ffd400", markeredgecolor=TINTA, markersize=15, linestyle="none")
ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3]); sin_ejes(ax)
ax.set_title(f"Dónde cayeron los focos, sobre la imagen HV del {dm(REF)} (estrella = pueblo)")
ax.legend(title="Fecha del foco", loc="lower left", frameon=True, facecolor="white", edgecolor=REJILLA)
plt.tight_layout(); plt.show()

**Lo que esto implica para el radar.** Más de la mitad de los focos de la caja son **anteriores al 17 de
agosto**, es decir, anteriores a nuestra primera imagen de radar limpia. Esas quemas ya habían ocurrido cuando
empieza la serie útil: el radar **no puede verlas como cambio**, porque no tiene un «antes» válido. El análisis
de abajo solo puede detectar lo ocurrido entre mediados de agosto y mediados de octubre.

# B. Análisis del MVP (exploratorio)

El MVP plantea cuatro cosas para esta región:

```
ANÁLISIS REAL DEL MVP
├── Incendios y cicatrices de quema                       → sección 9
├── Desmonte o pérdida de estructura vegetal              → sección 10
├── Recuperación / cambio de vegetación como contexto     → sección 11
└── Clasificación honesta: quema / pérdida previa /
    ambiguo / insuficiente                                → sección 10
```

Se sigue el orden del documento maestro (§12) con lo que hay. Dónde se aparta, y por qué:

| Paso del documento | Aquí | Diferencia |
|---|---|---|
| MC-01 · pila en lineal, ~60 m, con máscara | igual | — |
| MC-02 · línea base = mediana de **≥ 3** fechas previas | mediana de **2** fechas (17 y 29 ago) | Solo hay 5 fechas útiles; con 3 de línea base quedarían 2 para detectar |
| MC-05 · corrección por fenología con polígonos de control | se resta el cambio mediano de la **mitad más vegetada de la caja** | Los polígonos de control no están elegidos (DP-05) |
| MC-03 · dos umbrales **calibrados** | −2 dB (sensible) y −3 dB (estricto) | Son números redondos, sin calibrar |
| MC-04 · parches ≥ 1 ha, fechados con focos | igual (≥ 3 píxeles de 60 m = 1,08 ha) | — |
| MC-06 · una de cuatro clases por parche | igual, con una regla simple de orden temporal | La forma del parche se informa, no decide |
| MC-07 · hectáreas con rango | igual | El rango sale de los dos umbrales sin calibrar |

## 8. Preparación de la serie y línea base

In [ ]:
LINEA_BASE, OBJETIVO = BUENAS[:2], BUENAS[2:]
HA_PX = (tr3.a * abs(tr3.e)) / 1e4                              # hectáreas por píxel de 60 m

def valido60(f):
    # Píxel de 60 m válido solo si sus 3×3 píxeles de 20 m tienen mask = 1
    with rasterio.open(ruta(f, "mask")) as ds:
        m = ds.read(1)
    h, w = hh.shape
    return (m[: h * 3, : w * 3] == 1).reshape(h, 3, w, 3).all(axis=(1, 3))

HV = {f: leer(f, "HVHV", factor=3)[0] for f in BUENAS}          # potencia lineal, 60 m
VALIDO = np.logical_and.reduce([valido60(f) & np.isfinite(HV[f]) & (HV[f] > 0) for f in BUENAS])
base = np.where(VALIDO, np.median(np.stack([HV[f] for f in LINEA_BASE]), axis=0), np.nan)   # MC-02, en lineal
base_db = db(base)

# MC-03: cambio de HV en dB respecto de la línea base. MC-05 (sustituto): se resta el cambio mediano de la mitad
# más vegetada de la caja, para quitar lo que cambia en TODA la imagen (estación, humedad, calibración).
vegetado = VALIDO & (base_db >= np.nanmedian(base_db))
bruto = {f: np.where(VALIDO, db(HV[f]) - base_db, np.nan) for f in BUENAS}
deriva = {f: float(np.nanmedian(bruto[f][vegetado])) for f in BUENAS}
rel = {f: bruto[f] - deriva[f] for f in BUENAS}
ruido = 1.4826 * float(np.nanmedian(np.abs(rel[OBJETIVO[0]] - np.nanmedian(rel[OBJETIVO[0]]))))

print(f"Línea base: mediana de {', '.join(dm(f) for f in LINEA_BASE)} · fechas objetivo: {', '.join(dm(f) for f in OBJETIVO)}")
print(f"Píxeles válidos en las 5 fechas: {100 * VALIDO.mean():.1f}% de la caja ({VALIDO.sum() * HA_PX:,.0f} ha)")
print("Deriva de toda la escena que se resta (dB): " + " · ".join(f"{dm(f)} {deriva[f]:+.2f}" for f in BUENAS))
print(f"Dispersión típica del cambio en un píxel sin cambio real: ±{ruido:.2f} dB (estimación robusta)")

**Cómo leerlo.** La «deriva» es cuánto se movió toda la imagen de una fecha a otra sin que la tierra cambiara.
Restarla evita llamar cambio a la estación. La «dispersión» dice cuánto baila un píxel por el speckle: un
umbral de −2 dB queda a varias veces ese valor, así que el ruido solo rara vez lo cruza.

Los tres mapas muestran el cambio de HV de cada fecha objetivo respecto de la línea base. Marrón = la señal
bajó (posible pérdida de estructura); azul = subió.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 6.2))
prev = [LINEA_BASE[0]] + OBJETIVO[:-1]                            # inicio del intervalo de cada fecha objetivo
for ax, f, f0 in zip(axs, OBJETIVO, prev):
    im = ax.imshow(rel[f], cmap=CAMBIO, vmin=-4, vmax=4, extent=ext, interpolation="nearest")
    d = firms[(firms["acq_date"] > f0) & (firms["acq_date"] <= f)]
    ax.scatter(d["x"], d["y"], s=10, facecolors="none", edgecolors=TINTA, linewidths=0.6)
    ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3]); sin_ejes(ax)
    ax.set_title(f"{dm(f)} · {len(d)} focos desde el {dm(f0)} (círculos)")
fig.colorbar(im, ax=axs, orientation="horizontal", shrink=0.45, pad=0.04,
             label="cambio de HV respecto de la línea base (dB)   ◀ bajó · sin cambio · subió ▶")
fig.suptitle("Cambio de HV en cada fecha objetivo (gris oscuro = sin dato válido)", x=0.01, ha="left", fontsize=12)
plt.show()

## 9. Incendios y cicatrices de quema: ¿dónde y cuándo bajó HV?

Un píxel entra como **candidato a cicatriz** cuando:

1. su HV cae por debajo del umbral respecto de la línea base, **y**
2. la caída **se sostiene** en la fecha siguiente (al menos la mitad del umbral), para descartar speckle y
   lluvias pasajeras. En la última fecha no hay «siguiente», así que esas detecciones quedan **sin confirmar**;
3. forma un parche de **al menos 1 ha** (MC-04).

La **fecha de primera detección** es la primera fecha objetivo en que se cumple. No es la fecha del evento: el
evento ocurrió en algún momento de los 12 a 24 días anteriores.

Importante: esto detecta **caídas de HV**, no fuego. Que una caída sea quema, desmonte, cosecha u otra cosa se
discute en la sección 10.

In [ ]:
CRUZ = ndi.generate_binary_structure(2, 1); OCHO = np.ones((3, 3), bool)
MIN_PX = int(np.ceil(1 / HA_PX))                                  # 3 píxeles = 1,08 ha

def limpiar(m):
    m = ndi.binary_opening(m, structure=CRUZ)                     # quita píxeles sueltos
    etq, n = ndi.label(m, structure=OCHO)
    if n == 0: return m
    tam = ndi.sum(m, etq, index=np.arange(1, n + 1))
    return np.isin(etq, np.flatnonzero(tam >= MIN_PX) + 1)

def detectar(umbral):
    # Índice de la primera fecha objetivo con caída sostenida (−1 = sin detección)
    primera = np.full(hh.shape, -1)
    for i, f in enumerate(OBJETIVO):
        d = rel[f] < umbral
        if i < len(OBJETIVO) - 1:
            d &= rel[OBJETIVO[i + 1]] < umbral / 2                # la caída se sostiene
        d = limpiar(np.nan_to_num(d, nan=0).astype(bool))
        primera[(primera < 0) & d] = i
    return primera

U_SENSIBLE, U_ESTRICTO = -2.0, -3.0
prim_s, prim_e = detectar(U_SENSIBLE), detectar(U_ESTRICTO)

COL_FECHA = ["#f2b53a", "#d9641e", "#7a1f1f"]                     # de la primera a la última detección
fig, ax = plt.subplots(figsize=(10, 9.4))
ax.imshow(base_db, cmap=GRIS, vmin=lo2, vmax=hi2, extent=ext, alpha=0.55)
def agrandar(a, n=3):
    # Solo para dibujar: engrosa los parches n píxeles para que se vean a escala de toda la caja
    return ndi.grey_dilation(a, footprint=np.ones((2 * n + 1, 2 * n + 1), bool))
capa = np.ma.masked_less(agrandar(prim_s), 0)
ax.imshow(capa, cmap=ListedColormap(COL_FECHA), vmin=0, vmax=2, extent=ext, interpolation="nearest")
d = firms[(firms["acq_date"] > LINEA_BASE[0]) & (firms["acq_date"] <= OBJETIVO[-1])]
ax.scatter(d["x"], d["y"], s=16, facecolors="none", edgecolors=AZUL, linewidths=0.9)
ax.plot(px60, py60, marker="*", color="#ffd400", markeredgecolor=TINTA, markersize=15, linestyle="none")
ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3]); sin_ejes(ax)
etiquetas = [f"primera detección {dm(f)}" + (" (sin confirmar)" if i == len(OBJETIVO) - 1 else "") for i, f in enumerate(OBJETIVO)]
ax.legend(handles=[Patch(color=c, label=e) for c, e in zip(COL_FECHA, etiquetas)] +
          [plt.Line2D([], [], marker="o", markerfacecolor="none", markeredgecolor=AZUL, linestyle="none", label=f"focos {dm(LINEA_BASE[0])} – {dm(OBJETIVO[-1])} ({len(d)})"),
           plt.Line2D([], [], marker="*", color="#ffd400", markeredgecolor=TINTA, linestyle="none", markersize=12, label="San Ignacio de Velasco")],
          loc="lower left", frameon=True, facecolor="white", edgecolor=REJILLA)
ax.set_title(f"Candidatos a cicatriz (caída sostenida de HV > {abs(U_SENSIBLE):.0f} dB), dibujados engrosados 180 m para que se vean")
plt.tight_layout(); plt.show()

In [ ]:
# MC-07: hectáreas acumuladas por fecha, con su rango de sensibilidad (umbral estricto → sensible)
filas = []
for i, f in enumerate(OBJETIVO):
    filas.append({"fecha": f, "ha (umbral estricto, −3 dB)": ((prim_e >= 0) & (prim_e <= i)).sum() * HA_PX,
                  "ha (umbral sensible, −2 dB)": ((prim_s >= 0) & (prim_s <= i)).sum() * HA_PX,
                  "estado": "sin confirmar: falta una fecha posterior" if i == len(OBJETIVO) - 1 else "caída sostenida en la fecha siguiente"})
hect = pd.DataFrame(filas).set_index("fecha")

fig, ax = plt.subplots(figsize=(9, 4.4))
x = np.arange(len(OBJETIVO)); inf, sup = hect.iloc[:, 0].values, hect.iloc[:, 1].values
ax.vlines(x, inf, sup, color=NARANJA, linewidth=10, alpha=0.4)
ax.scatter(x, sup, s=80, color=NARANJA, zorder=3, edgecolors="white", linewidths=2)
ax.scatter(x, inf, s=80, color="#7a1f1f", zorder=3, edgecolors="white", linewidths=2)
for i in x:
    ax.annotate(f"{inf[i]:,.0f} – {sup[i]:,.0f} ha", (i, sup[i]), xytext=(14, 0), textcoords="offset points", va="center")
ax.annotate("umbral sensible (−2 dB)", (x[0], sup[0]), xytext=(-12, 10), textcoords="offset points", ha="right", color=TENUE, fontsize=9)
ax.annotate("umbral estricto (−3 dB)", (x[0], inf[0]), xytext=(-12, -14), textcoords="offset points", ha="right", color=TENUE, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels([dm(f) + ("\n(sin confirmar)" if i == len(OBJETIVO) - 1 else "") for i, f in enumerate(OBJETIVO)])
ax.set_xlim(-0.9, len(x) - 0.2); ax.set_ylim(0, sup.max() * 1.25); ax.set_ylabel("hectáreas acumuladas con caída de HV"); ax.grid(axis="y")
ax.set_title("Hectáreas acumuladas por fecha, con su rango de sensibilidad")
plt.tight_layout(); plt.show()
display(hect.style.format({c: "{:,.0f}" for c in hect.columns[:2]}))
print(f"Como referencia: la caja válida mide {VALIDO.sum() * HA_PX:,.0f} ha.")

**Lo que salió.** Los parches son **pocos y pequeños**: entre unas 300 y 600 ha según el umbral, sobre una caja
válida de unas 247.000 ha (cerca del 0,2 %). En el mapa están engrosados para poder verlos; su tamaño real es
el de la tabla.

**Cómo leer las hectáreas.** Son hectáreas con **caída sostenida de HV**, no hectáreas quemadas. El rango entre
los dos umbrales muestra cuánto depende la cifra de una decisión que todavía no está calibrada (RN-06): si el
rango es ancho, la cifra es frágil. La última fecha se informa aparte porque no pudo confirmarse.

Y recuerda el límite de la sección 7: aquí no entra nada de lo quemado antes del 17 de agosto.

## 10. ¿Quema o desmonte? Clasificación honesta de cada parche

La hipótesis del proyecto (§11.2) es de **orden temporal**:

- En un **incendio**, la caída de HV y los focos de calor llegan **juntos**.
- En un **desmonte**, HV cae **primero** y los focos llegan semanas **después**, cuando se quema lo tumbado.

Con eso, cada parche recibe **una de cuatro clases, nunca una causa confirmada** (RN-05). La regla usada aquí:

| Clase | Regla en este cuaderno |
|---|---|
| **Datos insuficientes** | El parche se detectó solo en la última fecha (sin confirmar), o toca el borde sin dato |
| **Compatible con quema** | Hay al menos un foco a menos de ~360 m del parche **dentro del intervalo** en que cayó HV |
| **Compatible con pérdida previa de estructura** | Ningún foco en ese intervalo, pero sí **después** de la caída |
| **Evidencia ambigua** | Cayó HV y no hay focos cerca, o solo los hay de antes |

Los ~360 m son el tamaño aproximado de un píxel de VIIRS: un foco no se puede ubicar con más precisión.

La **forma** del parche (qué tan rectangular es) se calcula y se muestra como evidencia de apoyo, pero no decide
la clase: en esta zona casi todo tiene bordes rectos, porque es tierra agrícola.

In [ ]:
etq, n_parches = ndi.label(prim_s >= 0, structure=OCHO)
ids = np.arange(1, n_parches + 1)

# Foco → parche más cercano, si está a ≤ 6 píxeles (360 m)
dist, (ir, ic) = ndi.distance_transform_edt(etq == 0, return_indices=True)
dentro = firms[(firms["fil"].between(0, hh.shape[0] - 1)) & (firms["col"].between(0, hh.shape[1] - 1))].copy()
f_, c_ = dentro["fil"].values, dentro["col"].values
dentro["parche"] = np.where(dist[f_, c_] <= 6, etq[ir[f_, c_], ic[f_, c_]], 0)

borde = ndi.binary_dilation(~VALIDO, structure=OCHO, iterations=2)
inicio = [pd.Timestamp(LINEA_BASE[0])] + [pd.Timestamp(f) for f in OBJETIVO[:-1]]     # inicio del intervalo de cada detección
CLASES = ["Compatible con quema", "Compatible con pérdida previa de estructura", "Evidencia ambigua", "Datos insuficientes"]
COL_CLASE = dict(zip(CLASES, ["#c8501e", "#6b4fa0", "#c9a227", "#8a939e"]))

filas = []
for geom, val in features.shapes(etq.astype("int32"), mask=etq > 0, transform=tr3, connectivity=8):
    pid = int(val); m = etq == pid
    i = int(np.bincount(prim_s[m]).argmax())                       # fecha de primera detección dominante
    fd = dentro[dentro["parche"] == pid]["acq_date"]
    t0, t1 = inicio[i], pd.Timestamp(OBJETIVO[i])
    n_antes, n_durante, n_despues = int((fd <= t0).sum()), int(((fd > t0) & (fd <= t1)).sum()), int((fd > t1).sum())
    if i == len(OBJETIVO) - 1 or (m & borde).any(): clase = CLASES[3]
    elif n_durante > 0: clase = CLASES[0]
    elif n_despues > 0: clase = CLASES[1]
    else: clase = CLASES[2]
    pol = geom_shape(geom)
    filas.append({"parche": pid, "clase": clase, "ha": m.sum() * HA_PX, "primera detección": OBJETIVO[i],
                  "caída media (dB)": float(np.nanmean(rel[OBJETIVO[i]][m])), "focos antes": n_antes,
                  "focos durante": n_durante, "focos después": n_despues,
                  "rectangularidad": pol.area / pol.minimum_rotated_rectangle.area})
parches = pd.DataFrame(filas)
if len(parches):
    parches = parches.groupby("parche").agg({"clase": "first", "ha": "first", "primera detección": "first", "caída media (dB)": "first",
              "focos antes": "first", "focos durante": "first", "focos después": "first", "rectangularidad": "max"}).reset_index()
mapa_clase = np.full(hh.shape, -1)
for k, cl in enumerate(CLASES):
    mapa_clase[np.isin(etq, parches.loc[parches["clase"] == cl, "parche"])] = k

fig, ax = plt.subplots(figsize=(10, 9.4))
ax.imshow(base_db, cmap=GRIS, vmin=lo2, vmax=hi2, extent=ext, alpha=0.5)
ax.imshow(np.ma.masked_less(agrandar(mapa_clase), 0), cmap=ListedColormap([COL_CLASE[c] for c in CLASES]), vmin=0, vmax=3, extent=ext, interpolation="nearest")
ax.plot(px60, py60, marker="*", color="#ffd400", markeredgecolor=TINTA, markersize=15, linestyle="none")
ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3]); sin_ejes(ax)
res = parches.groupby("clase").agg(parches=("parche", "size"), ha=("ha", "sum")).reindex(CLASES).fillna(0)
ax.legend(handles=[Patch(color=COL_CLASE[c], label=f"{c} · {int(res.loc[c, 'parches'])} parches · {res.loc[c, 'ha']:,.0f} ha") for c in CLASES],
          loc="lower left", frameon=True, facecolor="white", edgecolor=REJILLA)
ax.set_title("Cada parche con su clase (candidatas, nunca causas confirmadas); parches engrosados 180 m")
plt.tight_layout(); plt.show()

res["% de las ha"] = 100 * res["ha"] / res["ha"].sum()
res["rectangularidad mediana"] = parches.groupby("clase")["rectangularidad"].median().reindex(CLASES)
display(res.style.format({"parches": "{:.0f}", "ha": "{:,.0f}", "% de las ha": "{:.0f}%", "rectangularidad mediana": "{:.2f}"}))
print(f"Total: {len(parches)} parches, {parches['ha'].sum():,.0f} ha con el umbral sensible.")

In [ ]:
print("Los 12 parches más grandes:")
top = parches.sort_values("ha", ascending=False).head(12).set_index("parche")
display(top.style.format({"ha": "{:,.1f}", "caída media (dB)": "{:+.1f}", "rectangularidad": "{:.2f}"}))

Para ver si las clases se comportan distinto en el tiempo, esta figura sigue el cambio mediano de HV de los
píxeles de cada clase a lo largo de las cinco fechas limpias. La línea gris es la mitad más vegetada de la
caja, que por construcción queda en cero.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.8))
xs = [pd.Timestamp(f) for f in BUENAS]
ax.axvspan(xs[0], xs[len(LINEA_BASE) - 1], color="#f1f1f1"); ax.text(xs[0], 0.6, " línea base", color=TENUE, fontsize=9)
ax.plot(xs, [0] * len(xs), color=TENUE, linewidth=1.5, label="mitad más vegetada de la caja (referencia)")
for k, cl in enumerate(CLASES):
    m = mapa_clase == k
    if m.sum() == 0: continue
    y = [float(np.nanmedian(rel[f][m])) for f in BUENAS]
    ax.plot(xs, y, color=COL_CLASE[cl], linewidth=2, marker="o", markersize=7, label=cl)
ax.set_xticks(xs); ax.set_xticklabels([dm(f) for f in BUENAS]); ax.set_xlim(xs[0] - pd.Timedelta(days=2), xs[-1] + pd.Timedelta(days=3))
ax.set_ylabel("cambio mediano de HV (dB)"); ax.grid(axis="y")
ax.set_title("Trayectoria de HV de los parches de cada clase")
ax.legend(loc="lower left", fontsize=9); plt.tight_layout(); plt.show()

**Lo que salió.** En la corrida guardada hay 57 parches (590 ha con el umbral sensible): **más de la mitad de
las hectáreas quedan en «evidencia ambigua»** y un 11 % en «datos insuficientes». Solo 2 parches (51 ha) son
compatibles con quema y 10 (158 ha) con pérdida previa de estructura. La rectangularidad es casi igual en las
cuatro clases (0,61 a 0,67), así que aquí la forma no distingue nada.

La figura de trayectorias muestra además que los parches «compatibles con quema» ya venían bajando entre el
17 y el 29 de agosto, es decir, **dentro de la propia línea base**: con solo dos fechas de base, un evento de
fines de agosto la contamina. Es otra razón por la que el documento pide al menos tres fechas previas.

**Cómo leer la clasificación, con honestidad.**

- **«Compatible con» no significa «es».** Un parche «compatible con quema» es uno donde la caída de HV y un
  foco coincidieron en tiempo y lugar. Puede ser un incendio, o la quema de un chaqueo ya tumbado.
- **«Pérdida previa de estructura» es la clase más frágil.** Exige que los focos lleguen *después*, y nuestra
  serie termina el 16 de octubre: un desmonte cuyo material todavía no se quemó cae en «ambigua».
- **«Ambigua» es una respuesta válida, no un fallo.** En esta caja hay mucha agricultura mecanizada: una
  **cosecha** hace caer HV igual que un desmonte, sin fuego ni pérdida de bosque. Con solo radar y focos no se
  pueden separar. Haría falta una capa de cobertura (por ejemplo MapBiomas Bolivia) para saber qué era bosque.
- **«Insuficiente» incluye todo lo detectado en la última fecha**, porque no hubo una imagen posterior que lo
  confirmara.
- La rectangularidad va de 0 a 1 (1 = rectángulo perfecto). Si es parecida en todas las clases, la forma no
  ayuda a distinguirlas aquí, que es lo que se anticipaba.

## 11. Recuperación y cambio de vegetación como contexto

No todo cambio es pérdida. Dos miradas de contexto:

**El radar, hacia arriba.** Dónde HV **subió** de forma sostenida entre la línea base y el 16 de octubre.
Un aumento puede ser rebrote, cultivos creciendo o suelo más húmedo tras las primeras lluvias: es contexto, no
una medida de recuperación del bosque.

**El óptico, la estación.** El índice NBR de Sentinel-2 antes (7 jul) y después (28 sep). El bosque seco
chiquitano pierde hojas en invierno, así que una parte grande del «cambio» óptico de julio a septiembre es
**fenología**, no fuego. Por eso el dNBR sale positivo en casi toda la caja.

In [ ]:
ULT = OBJETIVO[-1]
sube = limpiar(np.nan_to_num((rel[ULT] > 2) & (rel[OBJETIVO[-2]] > 1), nan=0).astype(bool))
dnbr60 = nbr_antes60 - nbr_desp60

fig, axs = plt.subplots(1, 3, figsize=(17, 6.4))
axs[0].imshow(base_db, cmap=GRIS, vmin=lo2, vmax=hi2, extent=ext, alpha=0.5)
axs[0].imshow(np.ma.masked_equal(np.where(prim_s >= 0, 1, np.where(sube, 2, 0)), 0), cmap=ListedColormap(["#b5531c", AZUL]), vmin=1, vmax=2, extent=ext, interpolation="nearest")
axs[0].legend(handles=[Patch(color="#b5531c", label=f"HV bajó (candidatos, {(prim_s >= 0).sum() * HA_PX:,.0f} ha)"),
                       Patch(color=AZUL, label=f"HV subió más de 2 dB ({sube.sum() * HA_PX:,.0f} ha)")], loc="lower left", frameon=True, facecolor="white", edgecolor=REJILLA)
axs[0].set_title(f"Radar: dónde bajó y dónde subió HV hasta el {dm(ULT)}")
i1 = axs[1].imshow(nbr_desp60, cmap="Greens", vmin=-0.1, vmax=0.7, extent=ext); axs[1].set_title(f"Óptico: NBR del {dm(s2['despues']['fecha'])} (más verde = más vegetación)")
i2 = axs[2].imshow(dnbr60, cmap=CAMBIO.reversed(), vmin=-0.5, vmax=0.5, extent=ext); axs[2].set_title(f"Óptico: dNBR {dm(s2['antes']['fecha'])} → {dm(s2['despues']['fecha'])}")
fig.colorbar(i1, ax=axs[1], orientation="horizontal", pad=0.03, shrink=0.8, label="NBR")
fig.colorbar(i2, ax=axs[2], orientation="horizontal", pad=0.03, shrink=0.8, label="dNBR   ◀ más verde · sin cambio · vegetación perdida ▶")
for ax in axs: ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3]); sin_ejes(ax)
plt.tight_layout(); plt.show()
ok_d = np.isfinite(dnbr60)
print(f"dNBR mediano de la caja: {np.nanmedian(dnbr60):+.2f} · píxeles con dNBR > 0,10: {100 * np.mean(dnbr60[ok_d] > 0.10):.0f}% de la caja")

**Lectura.** Si la mayor parte de la caja tiene dNBR positivo, eso no puede ser todo fuego: es el bosque seco
perdiendo hojas entre julio y septiembre. Es la razón por la que el proyecto exige zonas de control (MC-05) y
por la que un umbral fijo de dNBR, sin corregir la estación, sobreestima lo quemado.

El borde gris inclinado alrededor de los mapas ópticos no es un error: Sentinel-2 está en la zona UTM 20S y el
radar en la 21S, y al pasar una grilla a la otra los bordes quedan girados.

## 12. Validación limitada

El documento pide comparar las cicatrices del radar con el dNBR de Sentinel-2 (VC-01) y usar los focos como
contexto (VC-02). Se hacen las dos cosas, sabiendo de antemano que **los periodos no coinciden**:

- el óptico cubre **7 jul → 28 sep**; el radar útil, **17 ago → 16 oct**;
- lo quemado entre el 7 de julio y el 17 de agosto está en el dNBR pero el radar no puede verlo;
- el dNBR incluye la caída de hojas de la estación.

Por eso esto se informa como **validación limitada** (RF-11), y un acuerdo bajo es un resultado que se reporta
tal cual.

In [ ]:
radar = prim_s >= 0
filas = []
for u in (0.27, 0.44):
    optico = dnbr60 > u; m = np.isfinite(dnbr60) & VALIDO
    vp = (radar & optico & m).sum(); fp = (radar & ~optico & m).sum(); fn = (~radar & optico & m).sum()
    filas.append({"referencia óptica": f"dNBR > {u:.2f}".replace(".", ","), "ha óptico": (optico & m).sum() * HA_PX, "ha radar": (radar & m).sum() * HA_PX,
                  "ha en común": vp * HA_PX, "precisión": vp / max(vp + fp, 1), "exhaustividad": vp / max(vp + fn, 1), "Dice": 2 * vp / max(2 * vp + fp + fn, 1)})
val = pd.DataFrame(filas).set_index("referencia óptica")
display(val.style.format({"ha óptico": "{:,.0f}", "ha radar": "{:,.0f}", "ha en común": "{:,.0f}", "precisión": "{:.2f}", "exhaustividad": "{:.2f}", "Dice": "{:.2f}"}))

con_foco = (parches[["focos antes", "focos durante", "focos después"]].sum(axis=1) > 0)
en_periodo = dentro[(dentro["acq_date"] > LINEA_BASE[0]) & (dentro["acq_date"] <= OBJETIVO[-1])]
print(f"VC-02 · parches con al menos un foco cerca (cualquier fecha): {con_foco.sum()} de {len(parches)} ({100 * con_foco.mean():.0f}%)")
print(f"        focos del periodo del radar que cayeron en o junto a un parche: {(en_periodo['parche'] > 0).sum()} de {len(en_periodo)} "
      f"({100 * (en_periodo['parche'] > 0).mean():.0f}%)")
print("Referencia del documento: Belenguer-Plomer et al. (2019) obtuvieron Dice 0,59 en banda C, con periodos emparejados.")

**Cómo leer la tabla.** *Precisión* = de lo que marcó el radar, cuánto también lo marca el óptico.
*Exhaustividad* = de lo que marca el óptico, cuánto encontró el radar. *Dice* resume las dos (1 = acuerdo
perfecto). Los cortes de dNBR (0,27 y 0,44) son las clases de severidad de uso general del USGS, no valores
calibrados para la Chiquitanía.

**Lo que salió.** La exhaustividad es de apenas 0,01 a 0,04 y el Dice de 0,01 a 0,07, muy lejos del 0,59 de
referencia. La precisión es alta (0,84 con dNBR > 0,27), pero ese corte lo supera cerca de un tercio de la caja
por la sola caída de hojas, así que dice poco. Y de los 191 focos del periodo del radar, **solo el 17 % cayó en
un parche o junto a él**: la mayoría de los fuegos detectados no produjo una caída de HV de 2 dB. Una
explicación posible, como **hipótesis**, es que fueran quemas de pasto o rastrojo, que dejan poca estructura
leñosa que perder en banda L.

Con periodos distintos y fenología sin corregir, estos números **no miden** la calidad del método: miden cuánto
se solapan dos cosas que no son la misma. Sirven para dimensionar el problema que el equipo de ciencia tiene
que resolver (fechas ópticas que encierren el periodo del radar y zonas de control).

# C. Cierre

## 13. Guía de imágenes raras

Resumen de todo lo que puede verse extraño en este cuaderno y qué es en realidad.

| Qué se ve | Dónde | Qué es | ¿Es un problema? |
|---|---|---|---|
| Grano fino, como arena | Todas las imágenes de radar | *Speckle*, propio del radar | No. Se reduce promediando |
| Cuña pálida en la esquina superior izquierda | Todas las de frame 169 | Borde del frame: ahí no hay datos | No. Esa franja la cubre el frame 170 |
| Imagen casi blanca con una banda | 18 jun | Valores ~30 veces demasiado altos; defecto del producto PROVISIONAL (la vista previa de NASA lo muestra igual) | Sí: fecha excluida |
| Un solo color plano | 12 jul | El archivo trae solo ceros | Sí: fecha excluida |
| Imagen casi blanca | 24 jul | Valores ~8 veces demasiado altos | Sí: fecha excluida |
| Imagen borrosa que «no cuadra» | 30 jun | Contenido real pero ubicado ~20 km fuera de lugar | Sí: fecha excluida |
| Campos a rayas | Falso color, HV | Parcelas con franjas alternas de vegetación (interpretación) | No |
| Mancha clara compacta junto a la estrella | HH, SWIR | El pueblo de San Ignacio de Velasco | No: confirma la ubicación |
| Toda la caja «perdió vegetación» | dNBR | Caída de hojas del bosque seco entre julio y septiembre | No es un error, pero impide leer el dNBR como fuego sin corregirlo |
| Borde gris girado | Mapas ópticos sobre la grilla del radar | Cambio de zona UTM (20S → 21S) | No |
| Parches con bordes muy rectos | Mapas de cambio | Parcelas agrícolas: cosecha, siembra o desmonte | No, pero el radar solo no dice cuál |

## 14. Qué aprendimos

**Sobre los datos**

- Los datos **sí apuntan a la región escogida**: el pueblo aparece donde dicen sus coordenadas y el radar limpio
  coincide con la imagen óptica sin desplazamiento (sección 2).
- **Solo las cinco fechas de radar posteriores al hueco (17 ago → 16 oct) son utilizables.** Las cuatro
  anteriores se excluyen: dos por valores, una vacía y el 30 de junio por estar desplazada unos 20 km. El README
  de la descarga todavía lo da por «utilizable»; hay que corregirlo.
- Hace falta comprobar la **posición**, no solo los valores, antes de admitir una fecha en la serie.
- El frame 169 cubre casi toda la caja; el 170 solo añade la franja norte y está en otra zona UTM.
- El significado de `mask` sigue pendiente (DP-03).

**Sobre el análisis del MVP**

- **Cicatrices de quema:** con cinco fechas se pueden detectar caídas sostenidas de HV entre mediados de agosto
  y mediados de octubre: unas 300 a 600 ha en 57 parches, cerca del 0,2 % de la caja. Solo el 17 % de los focos
  de ese periodo coincide con un parche. Las quemas de junio y julio (más de la mitad de los focos de la caja) quedan **fuera
  del alcance del radar** en esta órbita, por falta de un «antes» válido.
- **Quema o desmonte:** la regla de orden temporal funciona como procedimiento, pero la serie es corta y la
  agricultura confunde. Dos tercios de las hectáreas caen en «ambigua» o «insuficiente»: esa es la medida
  honesta de lo que se puede afirmar hoy.
- **Recuperación y contexto:** la caída de hojas domina el dNBR de julio a septiembre; sin zonas de control no
  se puede separar de la quema.
- **Validación:** limitada, por periodos que no coinciden; el acuerdo con el dNBR es muy bajo (Dice ≤ 0,07).

**Para decidir con el equipo**

1. Línea base con fechas posteriores al hueco y eventos de septiembre en adelante, **o** sumar otra órbita
   (la 140D tiene una fecha de julio con HH y HV y cubre casi toda la caja; su calidad está sin revisar).
2. Elegir los polígonos de control (DP-05) y calibrar los umbrales (MC-03).
3. Conseguir una imagen Sentinel-2 cercana al 17 de agosto, para que el par óptico encierre el periodo del radar.
4. Una capa de cobertura (bosque / no bosque) para no confundir cosecha con desmonte.

Todo lo de la parte B es exploratorio. Las cifras publicables deben salir del pipeline calibrado, con su rango
de sensibilidad y su validación.

**Créditos de datos.** NISAR L2 GCOV PROVISIONAL (NASA/ISRO, distribuido por Alaska Satellite Facility) ·
Focos de calor FIRMS (NASA LANCE: VIIRS S-NPP, VIIRS NOAA-20, MODIS) · Sentinel-2 L2A (ESA/Copernicus, vía
Earth Search) · Límites departamentales: geoBoundaries ADM1 (dominio público).